# STAIR-Enhanced v1: De-redundant Gated Projector (Module 1)

**Mục tiêu:** Thực nghiệm Ablation Study — Thay thế SVD Whitening tĩnh và tuyến tính trong STAIR bằng `DeRedundantGatedProjector` (Module 1: EMA Covariance + Null-space Projection + Gated Fusion).

**Cấu trúc dữ liệu & Pipeline:**
- 3 tập dữ liệu Amazon 2014 MMRec: **Baby**, **Sports**, **Electronics** (550-core)
- Cell 5: Huấn luyện **Baby + Sports** (cùng 1 cell)
- Cell 6: Huấn luyện **Electronics** (cell riêng biệt)
- Cell 7-9: Tự động trích xuất metric tại best validation epoch, so sánh với Baseline gốc và xuất bảng CSV/Biểu đồ cho Khóa luận.

**Số liệu Baseline gốc (Tái lập Table 2 - STAIR SIGIR 2025):**
| Tập dữ liệu | Recall@10 | Recall@20 | NDCG@10 | NDCG@20 | Best Epoch |
|---|---|---|---|---|---|
| Baby | 0.0674 | 0.1042 | 0.0359 | 0.0454 | 455/500 |
| Sports | 0.0743 | 0.1111 | 0.0405 | 0.0500 | 500/500 |
| Electronics | 0.0442 | 0.0665 | 0.0246 | 0.0303 | 490/500 |


In [ ]:
# Cell 1: Môi trường & Cài đặt Dependencies
import os, shutil, subprocess, sys

os.chdir('/kaggle/working')
repo = 'STAIR-Enhanced'
if os.path.exists(repo):
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git'], check=True)

# Cài đặt freerec và các thư viện cần thiết
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'freerec==0.9.7', 'torchdata==0.6.1', 'nvidia-ml-py', 'prettytable'], check=True)

import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_TAG  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric', '-f', f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_TAG}.html'], check=True)

import freerec
print('=' * 60)
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB')
print(f'FreeRec : {freerec.__version__}')
print('=' * 60)


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input sang /kaggle/data
import os, shutil

DATA_ROOT = '/kaggle/data'
os.makedirs(DATA_ROOT, exist_ok=True)

def copy_dataset(keywords, full_name):
    dest = os.path.join(DATA_ROOT, full_name)
    os.makedirs(dest, exist_ok=True)
    copied = []
    for root, _, files in os.walk('/kaggle/input'):
        if any(kw.lower() in root.lower() for kw in keywords):
            for f in files:
                if f.endswith(('.npy', '.pkl', '.txt', '.inter', '.item')):
                    shutil.copy(os.path.join(root, f), os.path.join(dest, f))
                    copied.append(f)
    print(f'[OK] {full_name}: {len(copied)} files copied.')

copy_dataset(['baby',        'amazon2014baby'],        'Amazon2014Baby_550_MMRec')
copy_dataset(['sports',      'amazon2014sports'],      'Amazon2014Sports_550_MMRec')
copy_dataset(['electronics', 'amazon2014electronics'], 'Amazon2014Electronics_550_MMRec')
print(f'\nToàn bộ dữ liệu đã sẵn sàng tại: {DATA_ROOT}')


In [ ]:
# Cell 3: Kiểm tra cấu hình và Modules trước khi chạy
import sys, os, yaml
sys.path.insert(0, '/kaggle/working/STAIR-Enhanced')

from models.enhanced_projector_v2 import DeRedundantGatedProjector
print('[1/2] Module DeRedundantGatedProjector import thành công!')

configs = [
    '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_550_MMRec.yaml',
    '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_550_MMRec.yaml',
    '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_550_MMRec.yaml',
]

print('[2/2] Kiểm tra file cấu hình:')
for cfg_path in configs:
    if os.path.exists(cfg_path):
        with open(cfg_path) as f:
            data = yaml.safe_load(f)
        print(f'  - {os.path.basename(cfg_path)}: dataset={data.get("dataset")}, epochs={data.get("epochs")}, monitors={data.get("monitors")}, which4best={data.get("which4best")}')
    else:
        print(f'  [ERR] Không tìm thấy {cfg_path}')


In [ ]:
# Cell 4: Hàm hỗ trợ chạy Training & Giám sát Phần cứng (VRAM Profiler)
import re, os, time, threading, subprocess
import pynvml

all_logs = {'baby': {'vram': [], 'time': []}, 'sports': {'vram': [], 'time': []}, 'electronics': {'vram': [], 'time': []}}
profiling_active = False
current_ds_key   = None

def hardware_profiler(interval=2.0):
    try:
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        t0 = time.time()
        while profiling_active:
            mem = pynvml.nvmlDeviceGetMemoryInfo(handle)
            if current_ds_key:
                all_logs[current_ds_key]['vram'].append(mem.used / 1024**2)
                all_logs[current_ds_key]['time'].append(time.time() - t0)
            time.sleep(interval)
        pynvml.nvmlShutdown()
    except Exception:
        pass

def extract_best_test(log_path):
    if not os.path.exists(log_path):
        return None, None
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        flat = f.read().replace('\n', ' ')
    ep_m = re.search(r'Load best model @Epoch:\s*(\d+)', flat)
    best_ep = int(ep_m.group(1)) if ep_m else None
    m = re.search(
        r'Load best model @Epoch.*?TEST.*?RECALL@10 Avg:\s*([0-9.]+).*?'
        r'RECALL@20 Avg:\s*([0-9.]+).*?NDCG@10 Avg:\s*([0-9.]+).*?NDCG@20 Avg:\s*([0-9.]+)',
        flat
    )
    if m:
        return best_ep, dict(zip(['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20'], [float(x) for x in m.groups()]))
    return best_ep, None

def run_training(key, yaml_cfg, data_root, log_path, extra_args=None):
    global profiling_active, current_ds_key
    print(f'\n{"="*60}')
    print(f'BẮT ĐẦU HUẤN LUYỆN: {key.upper()}')
    print(f'Config : {yaml_cfg}')
    print(f'Log    : {log_path}')
    print('='*60)

    current_ds_key = key
    profiling_active = True
    prof_thread = threading.Thread(target=hardware_profiler, args=(2.0,), daemon=True)
    prof_thread.start()

    cmd = ['python', 'main_enhanced_v1.py', '--config', yaml_cfg, '--root', data_root]
    if extra_args:
        cmd.extend(extra_args)

    t0 = time.time()
    with open(log_path, 'w', encoding='utf-8') as logf:
        result = subprocess.run(cmd, stdout=logf, stderr=subprocess.STDOUT, cwd='/kaggle/working/STAIR-Enhanced')
    elapsed = time.time() - t0

    profiling_active = False
    prof_thread.join(timeout=5)

    if result.returncode != 0:
        print(f'[THẤT BẠI] Mã lỗi {result.returncode} (Thời gian: {elapsed/60:.1f} phút)')
        with open(log_path, errors='replace') as f:
            print('30 dòng log cuối cùng:')
            print(''.join(f.readlines()[-30:]))
    else:
        print(f'[HOÀN THÀNH] {key.upper()} trong {elapsed/60:.1f} phút')
        ep, metrics = extract_best_test(log_path)
        if metrics:
            print(f'  - Best checkpoint @Epoch: {ep}')
            for k, v in metrics.items():
                print(f'    * {k}: {v:.6f}')
    return result.returncode


In [ ]:
# Cell 5: Huấn luyện STAIR-Enhanced v1 trên 2 tập dữ liệu Baby & Sports
# Cấu hình theo YAML gốc: 500 epochs, AdamWSEvo optimizer, embedding_dim=64
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_enhanced_v1'
os.makedirs(LOG_DIR, exist_ok=True)

# 1. Huấn luyện Baby
run_training(
    key       = 'baby',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Baby_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/baby.log',
)
torch.cuda.empty_cache()

# 2. Huấn luyện Sports
run_training(
    key       = 'sports',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Sports_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/sports.log',
)
torch.cuda.empty_cache()

print('=' * 60)
print('ĐÃ HOÀN THÀNH HUẤN LUYỆN TẬP BABY & SPORTS!')
print('=' * 60)


In [ ]:
# Cell 6: Huấn luyện STAIR-Enhanced v1 trên tập Electronics (Tập lớn nhất ~1.7M tương tác)
# Chạy riêng biệt ở cell này để quản lý thời gian trên Kaggle GPU (T4 ước tính 4-5 tiếng)
import torch, os

STAIR_DIR = '/kaggle/working/STAIR-Enhanced'
DATA_ROOT = '/kaggle/data'
LOG_DIR   = '/kaggle/working/logs_enhanced_v1'
os.makedirs(LOG_DIR, exist_ok=True)

run_training(
    key       = 'electronics',
    yaml_cfg  = f'{STAIR_DIR}/configs/Amazon2014Electronics_550_MMRec.yaml',
    data_root = DATA_ROOT,
    log_path  = f'{LOG_DIR}/electronics.log',
)
torch.cuda.empty_cache()

print('=' * 60)
print('ĐÃ HOÀN THÀNH HUẤN LUYỆN TẬP ELECTRONICS!')
print('=' * 60)


In [ ]:
# Cell 7: Tổng hợp kết quả và So sánh Ablation Study với Baseline gốc
from prettytable import PrettyTable
import os

LOG_DIR = '/kaggle/working/logs_enhanced_v1'

BASELINE = {
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

results = {}
for ds in ['baby', 'sports', 'electronics']:
    log = os.path.join(LOG_DIR, f'{ds}.log')
    ep, metrics = extract_best_test(log)
    results[ds] = {'epoch': ep, 'metrics': metrics}

METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

print('=' * 85)
print('BẢNG SO SÁNH ABLATION STUDY: STAIR-Baseline vs STAIR-Enhanced v1 (Module 1)')
print('=' * 85)

for ds in ['baby', 'sports', 'electronics']:
    t = PrettyTable()
    t.field_names = ['Chỉ số', 'STAIR Baseline', 'STAIR-Enhanced v1', 'Chênh lệch (Delta %)']
    enh = results[ds]['metrics'] or {}
    bl  = BASELINE[ds]
    for m in METRICS:
        bl_val  = bl.get(m, float('nan'))
        enh_val = enh.get(m, float('nan'))
        if bl_val and enh_val:
            delta = f"{(enh_val - bl_val)/bl_val*100:+.2f}%"
        else:
            delta = 'N/A'
        t.add_row([m, f'{bl_val:.6f}', f'{enh_val:.6f}' if enh_val else 'N/A', delta])
    print(f'\nTập dữ liệu: {ds.upper()} (Best Epoch: {results[ds]["epoch"]})')
    print(t)
print('=' * 85)


In [ ]:
# Cell 8: Xuất kết quả ra file CSV phục vụ viết Báo cáo Khóa luận
import csv, os

OUT_CSV = '/kaggle/working/ablation_enhanced_v1.csv'
METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

rows = []
for ds in ['baby', 'sports', 'electronics']:
    bl  = BASELINE[ds]
    enh = results[ds]['metrics'] or {}
    ep  = results[ds]['epoch']
    for m in METRICS:
        bl_val  = bl.get(m)
        enh_val = enh.get(m)
        delta   = (enh_val - bl_val)/bl_val*100 if bl_val and enh_val else None
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'STAIR-Baseline', 'value': f'{bl_val:.6f}' if bl_val else '', 'best_epoch': ''})
        rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'STAIR-Enhanced-v1', 'value': f'{enh_val:.6f}' if enh_val else 'N/A', 'best_epoch': str(ep) if ep else 'N/A'})
        if delta is not None:
            rows.append({'dataset': ds.capitalize(), 'metric': m, 'model': 'Delta (%)', 'value': f'{delta:+.2f}%', 'best_epoch': ''})

with open(OUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['dataset', 'metric', 'model', 'value', 'best_epoch'])
    writer.writeheader()
    writer.writerows(rows)

print(f'[ĐÃ XUẤT FILE] {OUT_CSV}')


In [ ]:
# Cell 9: Vẽ biểu đồ đường học tập (Learning Curves) & Mức sử dụng VRAM
import matplotlib.pyplot as plt, re

def parse_learning_curve(log_path):
    if not os.path.exists(log_path): return {}
    with open(log_path, encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()
    epochs, r20s, n20s = [], [], []
    for line in lines:
        ep_m = re.search(r'VALID @Epoch:\s*(\d+)', line)
        r_m  = re.search(r'RECALL@20 Avg:\s*([0-9.]+)', line, re.IGNORECASE)
        n_m  = re.search(r'NDCG@20 Avg:\s*([0-9.]+)',   line, re.IGNORECASE)
        if ep_m and r_m and n_m:
            epochs.append(int(ep_m.group(1)))
            r20s.append(float(r_m.group(1)))
            n20s.append(float(n_m.group(1)))
    return {'epoch': epochs, 'recall20': r20s, 'ndcg20': n20s}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('STAIR-Enhanced v1: VALID Metrics qua từng Epoch', fontsize=14, fontweight='bold')
COLORS = {'baby': '#1f77b4', 'sports': '#ff7f0e', 'electronics': '#2ca02c'}

for ds in ['baby', 'sports', 'electronics']:
    log = os.path.join('/kaggle/working/logs_enhanced_v1', f'{ds}.log')
    curve = parse_learning_curve(log)
    if curve.get('epoch'):
        axes[0].plot(curve['epoch'], curve['recall20'], label=f'{ds.capitalize()} (Enhanced)', color=COLORS[ds])
        axes[1].plot(curve['epoch'], curve['ndcg20'],   label=f'{ds.capitalize()} (Enhanced)', color=COLORS[ds])

BL = {'baby': (0.1042, 0.0454), 'sports': (0.1111, 0.0500), 'electronics': (0.0665, 0.0303)}
for ds, (r20, n20) in BL.items():
    axes[0].axhline(r20, color=COLORS[ds], linestyle='--', alpha=0.6, label=f'{ds.capitalize()} Baseline')
    axes[1].axhline(n20, color=COLORS[ds], linestyle='--', alpha=0.6, label=f'{ds.capitalize()} Baseline')

axes[0].set_title('Recall@20 (Đường đứt nét = Baseline)'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Recall@20'); axes[0].grid(True, linestyle=':', alpha=0.5); axes[0].legend(fontsize=8)
axes[1].set_title('NDCG@20 (Đường đứt nét = Baseline)'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('NDCG@20'); axes[1].grid(True, linestyle=':', alpha=0.5); axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig('/kaggle/working/learning_curves_enhanced_v1.png', dpi=150)
plt.show()
print('[ĐÃ LƯU BIỂU ĐỒ] /kaggle/working/learning_curves_enhanced_v1.png')


## 🛠️ Checklist Tham Số Cần Tune (Hyperparameter Tuning)

| Tham số | Giá trị mặc định | Miền thử nghiệm gợi ý | Ý nghĩa & Tác động |
|---|---|---|---|
| `ema_decay` | `0.99` | `[0.95, 0.99, 0.999]` | Hệ số cập nhật ma trận hiệp phương sai EMA. Giá trị nhỏ cập nhật nhanh hơn (hợp với tập lớn như Electronics), giá trị lớn ổn định hơn (hợp với Baby). |
| `null_rank` | `16` | `[8, 12, 16, 20, 24]` | Số chiều thành phần chính bị loại bỏ trong Null-space Projection. Loại bỏ quá ít sẽ giữ lại nhiễu, loại bỏ quá nhiều (> 32) có thể mất thông tin hữu ích. |
| `lr` | `1e-3` | `[5e-4, 1e-3, 2e-3]` | Tốc độ học của mô hình. Nếu loss giảm chậm ở các epoch đầu, có thể nâng lên `2e-3`. |
| `weight_decay` | `Theo YAML` | `[0.1, 0.3, 0.5]` | Regularization cho trọng số (Baby dùng 0.3, Sports & Electronics dùng 0.1). |
| `batch_size` | `Theo YAML` | `[512, 1024, 2048, 4096]` | Baby/Sports dùng 1024, Electronics dùng 4096. Batch lớn giúp ma trận hiệp phương sai ước lượng chính xác hơn. |
